# Strategy scan: all rules x all assets

Run every strategy in `sysstrat` on all five assets, then look at the picture from three angles: per-asset Sharpe, the diversified equal-weight portfolio per strategy, and signal correlations.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sysstrat import (
    Asset, Capital, FixedRiskSizer, BacktestRunner, PortfolioRunner,
    BuyAndHoldStrategy, MACrossoverStrategy, EWMACStrategy, NormalisedTrendStrategy,
    BreakoutStrategy, AccelerationStrategy, SkewStrategy, MeanReversionStrategy,
    TimeSeriesMomentumStrategy, DonchianStrategy, BollingerMeanReversionStrategy,
    MACDStrategy, RSIMeanReversionStrategy, FixedSignalStrategy,
    cross_sectional_momentum, cross_sectional_reversal,
    load_simple_price_csv, print_comparison_table,
)

In [2]:
# Locate the research repo root (contains data/MCFTR.csv)
ROOT = Path.cwd()
while not (ROOT / "data" / "MCFTR.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print(f"Data dir: {DATA_DIR}")

Data dir: c:\Users\igorp\OneDrive\Документы\GitHub\futures_trading_strategies\data


In [3]:
INSTRUMENTS = {
    "MCFTR":  "MCFTR.csv",       # broad equity index
    "RGBITR": "RGBITR.csv",      # government bond index (total return)
    "GLDRUB": "GLDRUB_TOM.csv",  # gold, in RUB
    "CNYRUB": "CNYRUB_TOM.csv",  # CNY/RUB FX
    "USDRUB": "USDRUB.csv",      # USD/RUB FX
}

assets = {
    t: Asset(ticker=t, price_data=load_simple_price_csv(DATA_DIR / f),
             commission_rate=0.0004, slippage_rate=0.001)
    for t, f in INSTRUMENTS.items()
}

start = max(a.price_data.index.min() for a in assets.values())
end = min(a.price_data.index.max() for a in assets.values())
assets = {t: a.slice(start, end) for t, a in assets.items()}
print(f"Common period: {start.date()} -> {end.date()}")

Common period: 2013-10-21 -> 2026-02-13


In [4]:
capital = Capital(initial_capital=100_000)
sizer = FixedRiskSizer(risk_target=0.20, max_leverage=1.0)

In [5]:
STRATEGIES = {
    "Buy & Hold":          BuyAndHoldStrategy(),
    "MA Cross (10/50)":    MACrossoverStrategy(short_window=10, long_window=50),
    "MA Cross LS":         MACrossoverStrategy(short_window=10, long_window=50, mode="long_short"),
    "EWMAC (16/64)":       EWMACStrategy(),
    "EWMAC (32/128)":      EWMACStrategy(fast_window=32, slow_window=128),
    "Normalised Trend":    NormalisedTrendStrategy(),
    "Breakout (40)":       BreakoutStrategy(horizon=40),
    "Breakout (160)":      BreakoutStrategy(horizon=160),
    "Acceleration":        AccelerationStrategy(),
    "Skew":                SkewStrategy(),
    "Mean Reversion":      MeanReversionStrategy(),
    "TSMOM (252/21)":      TimeSeriesMomentumStrategy(),
    "Donchian (55/20)":    DonchianStrategy(),
    "Bollinger MR (40,2)": BollingerMeanReversionStrategy(),
    "MACD (12/26/9)":      MACDStrategy(),
    "RSI(2) MR":           RSIMeanReversionStrategy(),
}

## 1. Per-asset Sharpe (strategy x asset)

Each cell is a single-asset backtest, vol-targeted at 20%.

In [6]:
sharpe = {}
total_ret = {}
for sname, strategy in STRATEGIES.items():
    sharpe[sname] = {}
    total_ret[sname] = {}
    for t, a in assets.items():
        m = BacktestRunner(capital, a, sizer).run(strategy).metrics
        sharpe[sname][t] = round(m.sharpe_ratio, 2)
        total_ret[sname][t] = round(m.total_return_pct, 1)

sharpe_df = pd.DataFrame(sharpe).T[list(assets)]
total_ret_df = pd.DataFrame(total_ret).T[list(assets)]
sharpe_df

,MCFTR,RGBITR,GLDRUB,CNYRUB,USDRUB
Buy & Hold,0.58,0.83,1.02,0.49,0.50
MA Cross (10/50),0.49,1.55,1.02,0.79,0.81
MA Cross LS,-0.02,0.77,0.84,0.87,0.86
EWMAC (16/64),0.33,0.89,0.67,0.58,0.52
EWMAC (32/128),0.36,0.66,0.62,0.26,0.17
Normalised Trend,0.24,0.83,0.60,0.35,0.28
Breakout (40),0.12,1.02,0.79,0.81,0.78
Breakout (160),0.53,0.79,0.73,0.44,0.38
Acceleration,-0.29,0.59,0.26,-0.01,0.23
Skew,-0.10,0.13,0.08,-0.24,0.21


In [7]:
total_ret_df

,MCFTR,RGBITR,GLDRUB,CNYRUB,USDRUB
Buy & Hold,139.5,86.2,247.6,106.3,105.4
MA Cross (10/50),70.4,83.7,218.8,146.8,143.0
MA Cross LS,-4.2,79.6,202.0,189.8,181.0
EWMAC (16/64),64.8,86.7,150.7,122.0,107.7
EWMAC (32/128),74.4,66.2,145.7,57.4,36.6
Normalised Trend,48.5,83.9,149.4,81.7,61.0
Breakout (40),24.5,102.7,191.6,168.6,162.0
Breakout (160),104.0,79.9,170.4,92.6,76.8
Acceleration,-57.1,58.0,53.5,-2.9,45.6
Skew,-23.6,13.5,18.4,-52.9,42.7


## 2. Diversified equal-weight portfolio per strategy

Each strategy run across the basket and combined 1/N.

In [8]:
rows = []
for sname, strategy in STRATEGIES.items():
    reports = {t: BacktestRunner(capital, a, sizer).run(strategy) for t, a in assets.items()}
    p = PortfolioRunner(capital).run(reports)
    m = p.metrics
    rows.append({
        "strategy": sname,
        "total ret %": round(m.total_return_pct, 1),
        "vol %": round(m.annual_volatility_pct, 2),
        "sharpe": round(m.sharpe_ratio, 2),
        "sortino": round(m.sortino_ratio, 2),
        "max DD %": round(m.max_drawdown_pct, 1),
    })

port_summary = pd.DataFrame(rows).set_index("strategy")
port_summary

,total ret %,vol %,sharpe,sortino,max DD %
strategy,,,,,
Buy & Hold,136.3,8.26,1.34,2.07,-25.3
MA Cross (10/50),132.1,7.43,1.44,2.19,-17.2
MA Cross LS,129.6,10.31,1.02,1.59,-17.1
EWMAC (16/64),106.4,9.61,0.90,1.22,-18.9
EWMAC (32/128),76.1,9.48,0.65,0.86,-25.5
Normalised Trend,85.1,10.49,0.66,0.91,-38.6
Breakout (40),130.3,9.87,1.07,1.52,-14.6
Breakout (160),104.6,9.39,0.91,1.28,-17.4
Acceleration,19.9,8.85,0.18,0.20,-49.4


## 3. Signal correlation (MCFTR)

Rules with correlation well below 1 capture different behaviour and are candidates for combination.

In [9]:
d = assets["MCFTR"].price_data.to_frame(name="close")
signals = {name: strategy.generate_signals(d) for name, strategy in STRATEGIES.items()}

corr = pd.DataFrame({
    a: {b: round(signals[a].corr(signals[b]), 2) for b in signals}
    for a in signals
})
corr

c:\Users\igorp\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\igorp\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,Buy & Hold,MA Cross (10/50),MA Cross LS,EWMAC (16/64),EWMAC (32/128),Normalised Trend,Breakout (40),Breakout (160),Acceleration,Skew,Mean Reversion,TSMOM (252/21),Donchian (55/20),"Bollinger MR (40,2)",MACD (12/26/9),RSI(2) MR
Buy & Hold,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MA Cross (10/50),NaN,1.00,0.99,0.76,0.53,0.73,0.85,0.49,0.44,-0.17,0.34,0.20,0.77,-0.66,0.04,-0.06
MA Cross LS,NaN,0.99,1.00,0.77,0.53,0.73,0.86,0.49,0.44,-0.18,0.34,0.18,0.75,-0.67,0.04,-0.05
EWMAC (16/64),NaN,0.76,0.77,1.00,0.84,0.95,0.83,0.79,0.34,-0.27,0.36,0.44,0.79,-0.64,-0.02,-0.12
EWMAC (32/128),NaN,0.53,0.53,0.84,1.00,0.80,0.52,0.95,-0.00,-0.30,0.39,0.72,0.57,-0.36,-0.15,-0.05
Normalised Trend,NaN,0.73,0.73,0.95,0.80,1.00,0.78,0.76,0.27,-0.23,0.37,0.42,0.76,-0.60,-0.05,-0.09
Breakout (40),NaN,0.85,0.86,0.83,0.52,0.78,1.00,0.49,0.63,-0.15,0.21,0.15,0.86,-0.84,0.24,-0.19
Breakout (160),NaN,0.49,0.49,0.79,0.95,0.76,0.49,1.00,-0.05,-0.26,0.36,0.68,0.54,-0.34,-0.12,-0.06
Acceleration,NaN,0.44,0.44,0.34,-0.00,0.27,0.63,-0.05,1.00,-0.05,-0.05,-0.20,0.51,-0.62,0.44,-0.17
Skew,NaN,-0.17,-0.18,-0.27,-0.30,-0.23,-0.15,-0.26,-0.05,1.00,-0.09,-0.07,-0.21,0.16,0.02,0.02


## 4. Cross-sectional strategies

Basket-level rules: compute signals on the cross-section, wrap each column in `FixedSignalStrategy`, and run through the normal pipeline.

In [10]:
prices = pd.DataFrame({t: a.price_data for t, a in assets.items()}).dropna()
cs_mom = cross_sectional_momentum(prices)
cs_rev = cross_sectional_reversal(prices)

cs_results = {}
for name, signals_df in [("CS Momentum", cs_mom), ("CS Reversal", cs_rev)]:
    reports = {
        t: BacktestRunner(capital, assets[t], sizer).run(
            FixedSignalStrategy(signals_df[t], name=f"{name} {t}")
        )
        for t in assets
    }
    cs_results[name] = PortfolioRunner(capital).run(reports)

bh_reports = {t: BacktestRunner(capital, a, sizer).run(BuyAndHoldStrategy()) for t, a in assets.items()}
bh_port = PortfolioRunner(capital).run(bh_reports)

print_comparison_table(
    "Cross-sectional vs equal-weight Buy & Hold",
    {
        "CS Momentum": cs_results["CS Momentum"].metrics,
        "CS Reversal": cs_results["CS Reversal"].metrics,
        "B&H equal-wt": bh_port.metrics,
    },
)


┌───────────────────────────────┬─────────────┬─────────────┬──────────────┐
│              Cross-sectional vs equal-weight Buy & Hold             │
├───────────────────────────────┼─────────────┼─────────────┼──────────────┤
│Metric                         │ CS Momentum │ CS Reversal │ B&H equal-wt │
├───────────────────────────────┼─────────────┼─────────────┼──────────────┤
│ Period                                                              │
├───────────────────────────────┼─────────────┼─────────────┼──────────────┤
│Years                          │       12.31 │       12.31 │        12.31 │
├───────────────────────────────┼─────────────┼─────────────┼──────────────┤
│ Returns & Performance                                               │
├───────────────────────────────┼─────────────┼─────────────┼──────────────┤
│Total Return %                 │     -48.85% │    -209.48% │     +136.29% │
│CAGR %                         │      -3.97% │     -17.01% │      +11.07% │
│Gross Return